## QA Code Review Agent

Multiple models independently review a code submission across key quality dimensions: correctness, style, test coverage, edge cases, and maintainability. A judge model ranks the reviews, and a final model merges them into a single authoritative QA report.

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)

In [ ]:
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

for name, key, prefix_len in [
    ("OpenAI", openai_api_key, 8),
    ("Anthropic", anthropic_api_key, 7),
    ("Google", google_api_key, 2),
    ("DeepSeek", deepseek_api_key, 3),
    ("Groq", groq_api_key, 4),
]:
    if key:
        print(f"{name} API Key exists and begins {key[:prefix_len]}")
    else:
        print(f"{name} API Key not set")

### Code Input

Fill in the fields below and run the cell. You can also load code directly from a file.

In [ ]:
import ipywidgets as widgets
from IPython.display import display as ipy_display

language_input = widgets.Dropdown(
    options=["Python", "JavaScript", "TypeScript", "Java", "Go", "Rust", "C", "C++", "Ruby", "Other"],
    value="Python",
    description="Language:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px"),
)

context_input = widgets.Textarea(
    placeholder="Briefly describe what this code does and where it lives in the codebase...",
    description="Context:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="700px", height="70px"),
)

code_input = widgets.Textarea(
    placeholder="Paste your code here...",
    description="Code:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="700px", height="300px"),
)

file_path_input = widgets.Text(
    placeholder="Or enter a file path to load code from (optional)",
    description="File path:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),
)

load_file_button = widgets.Button(description="Load from file", button_style="info")
status_label = widgets.Label(value="")

def load_from_file(btn):
    path = file_path_input.value.strip()
    if not path:
        status_label.value = "No file path provided."
        return
    try:
        with open(path, "r") as f:
            code_input.value = f.read()
        status_label.value = f"Loaded: {path}"
    except FileNotFoundError:
        status_label.value = f"File not found: {path}"
    except Exception as e:
        status_label.value = f"Error: {e}"

load_file_button.on_click(load_from_file)

ipy_display(
    language_input,
    context_input,
    code_input,
    widgets.HBox([file_path_input, load_file_button]),
    status_label,
)

In [ ]:
language = language_input.value
context = context_input.value.strip()
code_under_review = code_input.value.strip()

if not code_under_review:
    raise ValueError("No code provided. Paste your code into the Code field above and re-run this cell.")

if not context:
    context = "No additional context provided."

print(f"Language : {language}")
print(f"Context  : {context}")
print(f"Code     : {len(code_under_review)} characters, {len(code_under_review.splitlines())} lines")

In [ ]:
qa_review_request = f"""You are a senior software engineer performing a thorough QA code review. 
Review the following {language} code and evaluate it across these 6 dimensions:

1. Correctness - Does the code do what it is supposed to do? Are there logic errors or bugs?
2. Security - Are there security vulnerabilities, unsafe practices, or exposed secrets?
3. Edge Cases and Error Handling - Are edge cases considered? Is error handling robust?
4. Code Quality and Readability - Is the code clean, well-named, and easy to follow?
5. Test Coverage Assessment - What tests are missing? What scenarios should be tested?
6. Recommendations - Prioritized list of specific, actionable improvements.

Context: {context}

Code under review:
```{language}
{code_under_review}
```

Be specific. Reference line numbers or function names where relevant. 
At the end of your review, give an overall quality score from 1 to 10."""

messages = [{"role": "user", "content": qa_review_request}]

### Collect Reviews from Multiple Models

In [ ]:
openai = OpenAI()
competitors = []
answers = []

In [ ]:
model_name = "gpt-4o-mini"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(f"### {model_name}\n\n" + answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
model_name = "claude-3-7-sonnet-latest"

claude = Anthropic()
response = claude.messages.create(model=model_name, messages=messages, max_tokens=2000)
answer = response.content[0].text

display(Markdown(f"### {model_name}\n\n" + answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.0-flash"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(f"### {model_name}\n\n" + answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
deepseek = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com/v1")
model_name = "deepseek-chat"

response = deepseek.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(f"### {model_name}\n\n" + answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
model_name = "llama-3.3-70b-versatile"

response = groq.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(f"### {model_name}\n\n" + answer))
competitors.append(model_name)
answers.append(answer)

### Combine All Reviews

In [ ]:
together = ""
for index, (competitor, answer) in enumerate(zip(competitors, answers)):
    together += f"# Review from competitor {index + 1} ({competitor})\n\n"
    together += answer + "\n\n"

print(f"Collected {len(competitors)} reviews from: {competitors}")

### Judge: Rank the Reviews

In [ ]:
judge_prompt = f"""You are judging a QA code review competition between {len(competitors)} models.

Each model was asked to review this code:

{qa_review_request}

Evaluate each review on:
- Depth and accuracy of findings
- Whether it caught the most critical issues (security, correctness)
- Quality and specificity of recommendations
- Clarity and usefulness to a developer

Rank the reviews from best to worst. Respond with JSON only, no markdown, no explanation:
{{"results": ["best competitor number", "second best", ...]}}

Here are the reviews:

{together}

JSON only:"""

judge_messages = [{"role": "user", "content": judge_prompt}]

In [ ]:
response = openai.chat.completions.create(
    model="o3-mini",
    messages=judge_messages,
)
results_raw = response.choices[0].message.content
print("Raw judge output:", results_raw)

In [ ]:
results_dict = json.loads(results_raw)
ranks = results_dict["results"]

print("Review ranking by quality:\n")
for index, result in enumerate(ranks):
    competitor = competitors[int(result) - 1]
    print(f"  Rank {index + 1}: {competitor}")

### Final Merge: Consolidated QA Report

In [ ]:
merge_prompt = f"""You have received QA code reviews of the same code from {len(competitors)} different models.

Original code context: {context}

Here are all the reviews:

{together}

Your task is to synthesize these into a single, definitive QA report. The report must:

1. Consolidate all findings, removing duplicates but preserving distinct insights.
2. Prioritize issues by severity: Critical > High > Medium > Low.
3. Note any conflicting assessments and resolve them with a clear rationale.
4. Include a corrected or improved version of the code where appropriate.

Structure the final report with these sections:
- Executive Summary
- Critical Issues (must fix before shipping)
- High Priority Issues
- Medium / Low Priority Issues
- Missing Test Cases
- Suggested Code Improvements (with example code)
- Overall Quality Score (1-10) with justification

Write this as a final engineering report. Be direct and specific."""

response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": merge_prompt}],
)
final_report = response.choices[0].message.content

display(Markdown("## Final QA Report\n\n" + final_report))

In [ ]:
output_path = "qa_report.md"

with open(output_path, "w") as f:
    f.write(f"# QA Code Review Report\n\n")
    f.write(f"**Language:** {language}\n")
    f.write(f"**Context:** {context}\n\n")
    f.write("---\n\n")
    f.write(final_report)

print(f"Report saved to {output_path}")